# Hospital Readmission Prediction

DMT project notebook. This notebook follows the same workflow as `train_model.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv("../dataset/diabetic_data.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nTarget distribution:")
print(df["readmitted"].value_counts())

## Preprocessing
`<30` is the positive class (readmitted within 30 days). `>30` and `NO` are treated as 0 for this binary classification task.

In [ ]:
df = df.replace("?", np.nan)
df["target"] = (df["readmitted"].astype(str).str.strip() == "<30").astype(int)
features = [
    "race","gender","age","admission_type_id","admission_source_id",
    "time_in_hospital","num_lab_procedures","num_procedures","num_medications",
    "number_outpatient","number_emergency","number_inpatient","number_diagnoses",
    "insulin","diabetesMed"
]
numeric_features = [
    "time_in_hospital","num_lab_procedures","num_procedures","num_medications",
    "number_outpatient","number_emergency","number_inpatient","number_diagnoses"
]
categorical_features = [
    "race","gender","age","admission_type_id","admission_source_id","insulin","diabetesMed"
]
X = df[features].copy()
y = df["target"].astype(int)
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")
print(X.shape, y.value_counts())

## EDA

In [ ]:
sns.countplot(x=df["target"])
plt.title("30-Day Readmission Distribution")
plt.show()

In [ ]:
sns.boxplot(data=df, x="target", y="time_in_hospital")
plt.title("Time in Hospital vs Readmission")
plt.show()

In [ ]:
sns.boxplot(data=df, x="target", y="number_inpatient")
plt.title("Previous Inpatient Visits vs Readmission")
plt.show()

## Train/Test Split and Model Comparison

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, max_depth=12, class_weight="balanced", n_jobs=-1, random_state=42),
}

results = []
for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob),
    })

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False)
results_df

## Next step
Run `python train_model.py` from the project root to create the final saved model bundle and launch the Streamlit app with `streamlit run app/app.py`.